In [3]:
import json

input_file = "/Users/koustavsarkar/Downloads/meta_Amazon_Fashion.jsonl"
output_file = "meta_Amazon_Fashion.jsonl"

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:
    for i, line in enumerate(infile):
        if i >= 500000:
            break
        outfile.write(line)

print("Saved first 100000 rows to", output_file)

Saved first 100000 rows to meta_Amazon_Fashion.jsonl


In [5]:
import pandas as pd
import numpy as np
import json
import re

# ==========================================================
# LOAD AMAZON FASHION METADATA JSONL
# ==========================================================
meta_file = "meta_Amazon_Fashion.jsonl"

meta_rows = []

with open(meta_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            meta_rows.append(json.loads(line))
        except:
            pass

meta_df = pd.DataFrame(meta_rows)

print(f"Rows Loaded: {len(meta_df)}")

# ==========================================================
# KEEP ONLY IMPORTANT COLUMNS
# ==========================================================
meta_keep_cols = [
    "parent_asin",
    "main_category",
    "title",
    "average_rating",
    "rating_number",
    "price",
    "store",
    "details",
    "bought_together"
]

meta_df = meta_df[meta_keep_cols]

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================
def get_detail_value(details, key):
    if isinstance(details, dict):
        return details.get(key, "Unknown")
    return "Unknown"

def detect_subcategory(title):
    title = str(title).lower()

    mapping = {
        "sock": "Socks",
        "shirt": "Shirts",
        "t-shirt": "Shirts",
        "tee": "Shirts",
        "pant": "Pants",
        "jean": "Jeans",
        "palazzo": "Pants",
        "legging": "Leggings",
        "shoe": "Shoes",
        "sneaker": "Shoes",
        "boot": "Shoes",
        "dress": "Dress",
        "hoodie": "Hoodie",
        "sweatshirt": "Hoodie",
        "watch": "Watch",
        "bag": "Bag",
        "wallet": "Wallet",
        "belt": "Belt",
        "sunglass": "Sunglasses",
        "jacket": "Jacket",
        "coat": "Jacket"
    }

    for key, value in mapping.items():
        if key in title:
            return value

    return "Other"

def detect_gender(title):
    title = str(title).lower()

    if "women" in title or "woman" in title or "ladies" in title:
        return "Women"
    elif "men" in title or "man" in title:
        return "Men"
    elif "girl" in title:
        return "Girls"
    elif "boy" in title:
        return "Boys"
    elif "kid" in title or "kids" in title:
        return "Kids"
    else:
        return "Unisex"

def extract_color(title):
    colors = [
        "Black", "White", "Blue", "Red", "Green", "Pink", "Grey",
        "Gray", "Yellow", "Orange", "Purple", "Brown", "Navy",
        "Beige", "Khaki", "Gold", "Silver"
    ]

    title = str(title).lower()

    for color in colors:
        if color.lower() in title:
            return color

    return np.random.choice(["Black", "Blue", "Grey", "White", "Navy"])

def extract_size(title):
    title = str(title).upper()

    patterns = [
        r"\bXXXL\b", r"\bXXL\b", r"\bXL\b", r"\bL\b",
        r"\bM\b", r"\bS\b", r"\bXS\b"
    ]

    for pattern in patterns:
        match = re.search(pattern, title)
        if match:
            return match.group(0)

    return np.random.choice(["S", "M", "L", "XL"])

def extract_style(title):
    title = str(title).lower()

    if any(word in title for word in ["casual", "daily", "crew"]):
        return "Casual"
    elif any(word in title for word in ["sport", "athletic", "gym", "running"]):
        return "Sports"
    elif any(word in title for word in ["formal", "office", "business"]):
        return "Formal"
    elif any(word in title for word in ["designer", "party", "fashion"]):
        return "Fashion"
    elif any(word in title for word in ["vintage", "retro"]):
        return "Vintage"
    else:
        return "Regular"

def detect_fashion_type(title):
    title = str(title).lower()

    if any(word in title for word in ["sport", "athletic", "running", "gym"]):
        return "Sportswear"
    elif any(word in title for word in ["casual", "daily", "crew"]):
        return "Casual"
    elif any(word in title for word in ["formal", "office"]):
        return "Formal"
    elif any(word in title for word in ["designer", "party"]):
        return "Luxury"
    else:
        return "General"

def detect_season(title):
    title = str(title).lower()

    if any(word in title for word in ["hoodie", "jacket", "coat", "warm"]):
        return "Winter"
    elif any(word in title for word in ["dry fit", "lightweight", "summer"]):
        return "Summer"
    elif any(word in title for word in ["rain", "waterproof"]):
        return "Monsoon"
    else:
        return "All Season"

# ==========================================================
# DERIVED PRODUCT COLUMNS
# ==========================================================
meta_df["brand"] = meta_df["store"].fillna("Unknown Brand")
meta_df["subcategory"] = meta_df["title"].apply(detect_subcategory)
meta_df["gender"] = meta_df["title"].apply(detect_gender)

meta_df["material"] = meta_df["details"].apply(
    lambda x: get_detail_value(x, "Material")
)

meta_df["color"] = meta_df["details"].apply(
    lambda x: get_detail_value(x, "Color")
)

meta_df["size"] = meta_df["details"].apply(
    lambda x: get_detail_value(x, "Size")
)

meta_df["style"] = meta_df["details"].apply(
    lambda x: get_detail_value(x, "Style")
)

# ==========================================================
# FILL UNKNOWN VALUES
# ==========================================================
subcategory_material_map = {
    "Socks": "Cotton Blend",
    "Shirts": "Cotton",
    "Pants": "Polyester Blend",
    "Jeans": "Denim",
    "Leggings": "Spandex",
    "Shoes": "Leather",
    "Dress": "Rayon",
    "Hoodie": "Fleece",
    "Watch": "Stainless Steel",
    "Bag": "Canvas",
    "Wallet": "Leather",
    "Belt": "Leather",
    "Sunglasses": "Plastic",
    "Jacket": "Polyester",
    "Other": "Polyester"
}

meta_df["material"] = np.where(
    meta_df["material"].isin(["Unknown", "", None]),
    meta_df["subcategory"].map(subcategory_material_map),
    meta_df["material"]
)

meta_df["color"] = meta_df.apply(
    lambda row: extract_color(row["title"])
    if row["color"] in ["Unknown", "", None]
    else row["color"],
    axis=1
)

meta_df["size"] = meta_df.apply(
    lambda row: extract_size(row["title"])
    if row["size"] in ["Unknown", "", None]
    else row["size"],
    axis=1
)

meta_df["style"] = meta_df.apply(
    lambda row: extract_style(row["title"])
    if row["style"] in ["Unknown", "", None]
    else row["style"],
    axis=1
)

# ==========================================================
# PRICE FILLING LOGIC
# ==========================================================
subcategory_price_map = {
    "Socks": (8, 25),
    "Shirts": (15, 60),
    "Pants": (20, 80),
    "Jeans": (25, 90),
    "Leggings": (10, 40),
    "Shoes": (40, 150),
    "Dress": (25, 120),
    "Hoodie": (25, 90),
    "Watch": (50, 300),
    "Bag": (20, 150),
    "Wallet": (15, 80),
    "Belt": (10, 60),
    "Sunglasses": (15, 120),
    "Jacket": (30, 200),
    "Other": (10, 70)
}

def fill_price(row):
    if pd.notnull(row["price"]) and row["price"] not in ["", 0]:
        return float(row["price"])

    low, high = subcategory_price_map.get(row["subcategory"], (10, 50))
    return round(np.random.uniform(low, high), 2)

meta_df["price"] = meta_df.apply(fill_price, axis=1)

# ==========================================================
# PRODUCT FLAGS
# ==========================================================
meta_df["bought_together_flag"] = meta_df["bought_together"].apply(
    lambda x: 1 if pd.notnull(x) else 0
)

meta_df["luxury_flag"] = np.where(meta_df["price"] > 150, 1, 0)
meta_df["top_selling_flag"] = np.where(meta_df["rating_number"].fillna(0) >= 500, 1, 0)

meta_df["low_inventory_flag"] = np.random.choice(
    [0, 1],
    size=len(meta_df),
    p=[0.85, 0.15]
)

meta_df["inventory_status"] = np.where(
    meta_df["low_inventory_flag"] == 1,
    "Low Stock",
    "In Stock"
)

# ==========================================================
# PRODUCT AGE
# ==========================================================
meta_df["date_first_available"] = meta_df["details"].apply(
    lambda x: get_detail_value(x, "Date First Available")
)

meta_df["date_first_available"] = pd.to_datetime(
    meta_df["date_first_available"],
    errors="coerce"
)

current_year = pd.Timestamp.now().year

meta_df["product_age_years"] = current_year - meta_df["date_first_available"].dt.year

meta_df["product_age_bucket"] = pd.cut(
    meta_df["product_age_years"],
    bins=[-1, 1, 3, 5, 100],
    labels=["New Launch", "Recent", "Established", "Old"]
)

meta_df["fashion_type"] = meta_df["title"].apply(detect_fashion_type)
meta_df["seasonal_collection"] = meta_df["title"].apply(detect_season)

# ==========================================================
# BUSINESS METRICS
# ==========================================================
meta_df["impression_count"] = np.random.randint(1000, 50000, size=len(meta_df))
meta_df["view_count"] = np.random.randint(500, 25000, size=len(meta_df))
meta_df["cart_count"] = np.random.randint(50, 5000, size=len(meta_df))
meta_df["purchase_count"] = np.random.randint(10, 3000, size=len(meta_df))
meta_df["wishlist_count"] = np.random.randint(0, 5000, size=len(meta_df))

meta_df["click_through_rate"] = (
    meta_df["view_count"] / meta_df["impression_count"]
).round(2)

meta_df["conversion_rate"] = (
    meta_df["purchase_count"] / meta_df["view_count"]
).round(2)

meta_df["return_rate"] = np.round(np.random.uniform(0.01, 0.25, size=len(meta_df)), 2)
meta_df["profit_margin"] = np.round(np.random.uniform(0.10, 0.60, size=len(meta_df)), 2)

meta_df["gross_revenue"] = (
    meta_df["purchase_count"] * meta_df["price"]
).round(2)

meta_df["net_revenue"] = (
    meta_df["gross_revenue"] * (1 - meta_df["return_rate"])
).round(2)

meta_df["estimated_profit"] = (
    meta_df["net_revenue"] * meta_df["profit_margin"]
).round(2)

meta_df["discount_percentage"] = np.random.randint(5, 60, size=len(meta_df))

meta_df["discounted_price"] = (
    meta_df["price"] * (1 - meta_df["discount_percentage"] / 100)
).round(2)

meta_df["discount_loss"] = (
    meta_df["gross_revenue"] - meta_df["net_revenue"]
).round(2)

meta_df["engagement_score"] = (
    meta_df["view_count"] * 0.2 +
    meta_df["cart_count"] * 0.3 +
    meta_df["purchase_count"] * 0.5 +
    meta_df["wishlist_count"] * 0.1
).round(2)

meta_df["product_popularity_score"] = (
    meta_df["average_rating"].fillna(0) * 20 +
    np.log1p(meta_df["rating_number"].fillna(0)) * 10 +
    meta_df["purchase_count"] / 100
).round(2)

# ==========================================================
# MARKETING ATTRIBUTION
# ==========================================================
campaign_names = [
    "Summer Sale",
    "Festive Offer",
    "End of Season Sale",
    "New Arrival Push",
    "Influencer Promo",
    "Weekend Flash Sale"
]

meta_df["campaign_name"] = np.random.choice(campaign_names, size=len(meta_df))
meta_df["campaign_type"] = np.random.choice(
    ["Search Ad", "Display Ad", "Social Media", "Email", "Affiliate"],
    size=len(meta_df)
)

meta_df["ad_group"] = np.random.choice(
    ["Men Fashion", "Women Fashion", "Footwear", "Accessories"],
    size=len(meta_df)
)

meta_df["utm_source"] = np.random.choice(
    ["google", "facebook", "instagram", "youtube", "email"],
    size=len(meta_df)
)

meta_df["utm_medium"] = np.random.choice(
    ["cpc", "organic", "affiliate", "social", "email"],
    size=len(meta_df)
)

meta_df["utm_campaign"] = meta_df["campaign_name"].str.lower().str.replace(" ", "_")
meta_df["landing_page_type"] = np.random.choice(
    ["Homepage", "Product Page", "Category Page", "Offer Page"],
    size=len(meta_df)
)

meta_df["referral_channel"] = np.random.choice(
    ["Organic", "Paid Search", "Affiliate", "Direct", "Social"],
    size=len(meta_df)
)

meta_df["social_platform"] = np.random.choice(
    ["Instagram", "Facebook", "Pinterest", "YouTube", "TikTok"],
    size=len(meta_df)
)

meta_df["influencer_campaign_flag"] = np.random.choice(
    [0, 1],
    size=len(meta_df),
    p=[0.8, 0.2]
)

# ==========================================================
# LLM / RAG FIELDS
# ==========================================================
meta_df["review_quality_score"] = np.round(
    np.random.uniform(0.5, 1.0, size=len(meta_df)),
    2
)

meta_df["return_risk_level"] = np.where(
    meta_df["return_rate"] > 0.18,
    "High",
    np.where(meta_df["return_rate"] > 0.10, "Medium", "Low")
)

meta_df["sales_performance"] = np.where(
    meta_df["purchase_count"] > 1500,
    "High Performing",
    np.where(meta_df["purchase_count"] > 500, "Moderate", "Low")
)

meta_df["customer_interest_level"] = np.where(
    meta_df["wishlist_count"] > 3000,
    "Very High",
    np.where(meta_df["wishlist_count"] > 1000, "Medium", "Low")
)

meta_df["product_summary"] = (
    meta_df["brand"].fillna("Unknown Brand") + " " +
    meta_df["subcategory"].fillna("Fashion Product") +
    " for " + meta_df["gender"].fillna("Unisex") +
    " made from " + meta_df["material"].fillna("Mixed Material") +
    " in " + meta_df["color"].fillna("Multiple") + " color."
)

meta_df["top_keywords"] = meta_df.apply(
    lambda row: ", ".join([
        str(row["subcategory"]),
        str(row["material"]),
        str(row["color"]),
        str(row["fashion_type"]),
        str(row["seasonal_collection"])
    ]),
    axis=1
)

# ==========================================================
# FINAL OUTPUT
# ==========================================================
meta_df = meta_df.drop(columns=["details", "bought_together"], errors="ignore")

meta_df.to_csv("fashion_metadata_final_enriched.csv", index=False)
meta_df.to_json("fashion_metadata_final_enriched.json", orient="records", lines=True)

print(meta_df.head())
print(meta_df.columns)

Rows Loaded: 500000


/var/folders/vw/x71ctk413sdd79zvc97fm81w0000gn/T/ipykernel_25053/955347651.py:479: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  meta_df.to_json("fashion_metadata_final_enriched.json", orient="records", lines=True)


  parent_asin   main_category  \
0  B08BHN9PK5  AMAZON FASHION   
1  B08R39MRDW  AMAZON FASHION   
2  B077KJHCJ4  AMAZON FASHION   
3  B0811M2JG9  AMAZON FASHION   
4  B07SB2892S  AMAZON FASHION   

                                               title  average_rating  \
0  YUEDGE 5 Pairs Men's Moisture Control Cushione...             4.6   
1  DouBCQ Women's Palazzo Lounge Wide Leg Casual ...             4.1   
2  Pastel by Vivienne Honey Vanilla Girls' Trapez...             4.3   
3                                   Mento Streamtail             2.0   
4  RONNOX Women's 3-Pairs Bright Colored Calf Com...             4.3   

   rating_number  price               store               brand subcategory  \
0             16  10.47            GiveGift            GiveGift       Socks   
1              7  57.46              DouBCQ              DouBCQ       Pants   
2             11  68.75  Pastel by Vivienne  Pastel by Vivienne       Dress   
3              1  29.81          Guy Harvey         